<a href="https://colab.research.google.com/github/r021n/simple-machine-learing/blob/main/Spam_vs_Ham_Email_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1: Data Loading dan Eksplorasi

##  Import Library dan Download Dataset

In [1]:
import pandas as pd
import requests
import zipfile
import io

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

# Download the ZIP file content
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

# The desired file name inside the zip
csv_file_name = 'SMSSpamCollection'

# Read the specific file from the zip archive
with zip_file.open(csv_file_name) as f:
    df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Melihat Bentuk dan Info Data

In [2]:
print("Shape:", df.shape)

print("\nInfo:")
df.info()

Shape: (5572, 2)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


## Memeriksa Data Kosong dan Duplikat

In [3]:
print("Nilai kosong per kolom:")
print(df.isnull().sum())

print(f"\nJumlah data duplikat: {df.duplicated().sum()}")

df = df.drop_duplicates()
print(f"Shape setelah hapus duplikat: {df.shape}")

Nilai kosong per kolom:
label      0
message    0
dtype: int64

Jumlah data duplikat: 403
Shape setelah hapus duplikat: (5169, 2)


## Melihat Distribusi Label

In [4]:
print("Distribusi label:")
print(df['label'].value_counts())

print("\nPersentase:")
print(df['label'].value_counts(normalize=True).round(4) * 100)

Distribusi label:
label
ham     4516
spam     653
Name: count, dtype: int64

Persentase:
label
ham     87.37
spam    12.63
Name: proportion, dtype: float64


## Melihat Contoh Pesan Spam dan Ham

In [5]:
print("Contoh pesan HAM:")
print(df[df['label'] == 'ham']['message'].values[0])

print("\n" + "="*50)

print("\nContoh pesan SPAM:")
print(df[df['label'] == 'spam']['message'].values[0])

Contoh pesan HAM:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


Contoh pesan SPAM:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


# Fase 2: Text Preprocessing

## Import Library yang Dibutuhkan

In [6]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

print("Stopwords loaded:", len(stop_words), "kata")
print("Contoh stopwords:", list(stop_words)[:10])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Stopwords loaded: 198 kata
Contoh stopwords: ['where', 'on', 'yourselves', 'herself', 'only', "mustn't", 'can', 'her', 'needn', 'with']


## Membuat Fungsi Preprocessing

In [7]:
def preprocess_text(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]', '', text)

  words = text.split()
  words = [stemmer.stem(word) for word in words if word not in stop_words]

  return ' '.join(words)

contoh = "WINNER!! You have WON a £1000 prize! Call 09061234 NOW"
print("sebelum", contoh)
print("sesudah", preprocess_text(contoh))

sebelum WINNER!! You have WON a £1000 prize! Call 09061234 NOW
sesudah winner prize call
